실습 환경과 라이브러리 준비

In [1]:
%pip install -U transformers accelerate huggingface_hub sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 91.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 111.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 117.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [transformers] [transformers]ub]
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install -U transformers accelerate huggingface_hub sentencepiece pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 27.1 MB/s  0:00:00eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ==========================================
# 03. Gemma GPU LLM Practice
# Step 1. 실행 환경 확인
# ==========================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("GPU             : Not available (CPU mode)")

PyTorch version : 2.8.0+cu128
CUDA available  : True
GPU             : NVIDIA A100-SXM4-80GB


In [3]:
# ==========================================
# Step 5-1. Gemma 모델 저장 경로 설정
# ==========================================

import os

cache_dir = "/root/hf-cache"
os.makedirs(cache_dir, exist_ok=True)

print("Cache directory :", cache_dir)

Cache directory : /root/hf-cache


In [4]:
# ==========================================
# Step 5-2. Gemma 3 4B BF16 모델 로드
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir=cache_dir
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=cache_dir
)

print("\n===== Gemma 모델 로드 완료 =====")
print("Model :", model_id)
print("GPU :", torch.cuda.get_device_name(0))
print(
    "GPU Memory Allocated :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "GPU Memory Peak :",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]


===== Gemma 모델 로드 완료 =====
Model : google/gemma-3-4b-it
GPU : NVIDIA A100-SXM4-80GB
GPU Memory Allocated : 8.01 GB
GPU Memory Peak : 8.01 GB


In [5]:
# ==========================================
# Step 6. Gemma 3 4B 추론 성능 측정
# ==========================================

import time
import torch

prompt = "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."

messages = [
    {"role": "user", "content": [{"type": "text", "text": prompt}]}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Warm-up
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

input_tokens = inputs["input_ids"].shape[1]
generated_tokens = outputs.shape[1] - input_tokens
tokens_per_sec = generated_tokens / elapsed

response = tokenizer.decode(
    outputs[0][input_tokens:],
    skip_special_tokens=True
)

print("===== Gemma 3 4B 추론 결과 =====")
print(response)

print("\n===== 성능 측정 =====")
print("GPU :", torch.cuda.get_device_name(0))
print("Precision : BF16")
print("Generated tokens :", generated_tokens)
print("Generation time :", round(elapsed, 3), "sec")
print("Speed :", round(tokens_per_sec, 2), "tokens/sec")
print(
    "GPU Memory Allocated :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "GPU Memory Peak :",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

===== Gemma 3 4B 추론 결과 =====
생성형 AI는 기존 인공지능과 달리 데이터를 학습하여 새로운 콘텐츠를 창작해내는 반면, 기존 인공지능은 주어진 데이터를 분석하고 패턴을 인식하여 예측이나 분류와 같은 특정 작업을 수행합니다. 즉, 생성형 AI는 '만들기'에 집중하는 반면, 기존 인공지능은 '알기'에 집중한다고 볼 수 있습니다. 이러한 차이 때문에 생성형 AI

===== 성능 측정 =====
GPU : NVIDIA A100-SXM4-80GB
Precision : BF16
Generated tokens : 90
Generation time : 3.845 sec
Speed : 23.41 tokens/sec
GPU Memory Allocated : 8.02 GB
GPU Memory Peak : 8.04 GB


In [8]:
# ==========================================
# Step 7. Qwen3-8B vs Gemma 3 4B 비교
# ==========================================

import pandas as pd

results = pd.DataFrame({
    "Model": ["Qwen3-8B", "Gemma-3-4B-IT"],
    "Precision": ["BF16", "BF16"],
    "Generated Tokens": [90, 90],
    "Time (sec)": [3.691, 3.845],
    "Speed (tokens/sec)": [24.38, 23.41],
    "VRAM Allocated (GB)": [15.26, 8.02],
    "VRAM Peak (GB)": [15.28, 8.04]
})

results

,Model,Precision,Generated Tokens,Time (sec),Speed (tokens/sec),VRAM Allocated (GB),VRAM Peak (GB)
0,Qwen3-8B,BF16,90,3.691,24.38,15.26,15.28
1,Gemma-3-4B-IT,BF16,90,3.845,23.41,8.02,8.04
